# Facial Identification
Face detection and matching using InsightFace (buffalo_l model).

**How it works:**
1. You provide a **reference image** (the known person's photo)
2. You provide a **test image** (the image to search for matching faces)
3. The model detects all faces in both images and compares embeddings using cosine similarity
4. If similarity >= threshold, the face is marked as a **MATCH**

In [ ]:
!pip install insightface onnxruntime

In [ ]:
import insightface
import cv2
import numpy as np
import matplotlib.pyplot as plt
import os

In [ ]:
# Load InsightFace model (buffalo_l)
print("Loading model...")
model = insightface.app.FaceAnalysis(name='buffalo_l')
model.prepare(ctx_id=-1, det_size=(640, 640))
print("Model loaded successfully")

In [ ]:
def cosine_similarity(a, b):
    """Calculate cosine similarity between two embedding vectors."""
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))


def draw_faces(img, faces, matched_index=None):
    """Draw bounding boxes on detected faces. Green = matched, Red = others."""
    img_copy = img.copy()
    for i, f in enumerate(faces):
        x1, y1, x2, y2 = map(int, f.bbox)
        color = (0, 255, 0) if i == matched_index else (255, 0, 0)
        label = f"Face {i}"
        cv2.rectangle(img_copy, (x1, y1), (x2, y2), color, 2)
        cv2.putText(img_copy, label, (x1, y1 - 10),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.6, color, 2)
    plt.figure(figsize=(10, 8))
    plt.imshow(cv2.cvtColor(img_copy, cv2.COLOR_BGR2RGB))
    plt.axis("off")
    plt.show()

## Step 1: Load Reference Image
Provide the file path to the known person's photo (the reference face to match against).

In [ ]:
# ---- SET YOUR REFERENCE IMAGE PATH HERE ----
reference_image_path = input("Enter the path to the reference image: ").strip()

if not os.path.exists(reference_image_path):
    raise FileNotFoundError(f"Reference image not found: {reference_image_path}")

reference_img = cv2.imread(reference_image_path)
if reference_img is None:
    raise ValueError(f"Failed to read image: {reference_image_path}")

reference_faces = model.get(reference_img)

if len(reference_faces) == 0:
    raise ValueError("No face detected in reference image")

reference_emb = reference_faces[0].embedding
print(f"Face detected in reference image ({os.path.basename(reference_image_path)})")
draw_faces(reference_img, reference_faces)

## Step 2: Load Test Image
Provide the file path to the test image. This image can contain multiple faces — the model will find the best match.

In [ ]:
# ---- SET YOUR TEST IMAGE PATH HERE ----
test_image_path = input("Enter the path to the test image: ").strip()

if not os.path.exists(test_image_path):
    raise FileNotFoundError(f"Test image not found: {test_image_path}")

test_img = cv2.imread(test_image_path)
if test_img is None:
    raise ValueError(f"Failed to read image: {test_image_path}")

test_faces = model.get(test_img)

if len(test_faces) == 0:
    raise ValueError("No face detected in test image")

print(f"{len(test_faces)} face(s) detected in test image")
draw_faces(test_img, test_faces)

## Step 3: Compare Faces
Compare each detected face in the test image against the reference face using cosine similarity.

In [ ]:
# Compare all test faces against the reference embedding
similarities = []

for i, face in enumerate(test_faces):
    sim = cosine_similarity(reference_emb, face.embedding)
    similarities.append(sim)
    print(f"Face {i} similarity: {sim:.3f}")

best_index = int(np.argmax(similarities))
best_similarity = similarities[best_index]

In [ ]:
# Match threshold (lower = stricter, 0.45 is glare-tolerant)
threshold = 0.45

print("\n================ RESULT ================")

if best_similarity >= threshold:
    print(f"MATCH FOUND")
    print(f"  Matching Face Index : Face {best_index}")
    print(f"  Similarity Score    : {best_similarity:.3f}")
    print(f"\n  Green box = matched face")
    print(f"  Red boxes = other faces")
    draw_faces(test_img, test_faces, matched_index=best_index)
else:
    print("NO MATCH FOUND IN FRAME")
    print(f"  Best similarity: {best_similarity:.3f} (below threshold {threshold})")
    draw_faces(test_img, test_faces)